# Regression Task – Template Notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

RANDOM_STATE = 42


In [ ]:
def plot_residuals(y_true, y_pred, title):
    residuals = y_true - y_pred
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(0, linestyle="--")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Residuals")
    plt.show()

def regression_metrics(y_true, y_pred, n_features):
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_true) - 1) / (len(y_true) - n_features - 1)

    return {
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2,
        "Adjusted R2": adj_r2
    }

def train_test_comparison(model, y_train, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    n_features = model.named_steps["preprocessor"] \
        .transform(X).shape[1]

    metrics_df = pd.DataFrame.from_dict({
        "Train": regression_metrics(y_train, y_train_pred, n_features),
        "Test": regression_metrics(y_test, y_test_pred, n_features)
        }, orient="index")
    display(metrics_df)

def evaluate_model(name, model, X, y):
    y_pred = model.predict(X)

    # number of features after preprocessing
    n_features = model.named_steps["preprocessor"] \
        .transform(X).shape[1]

    metrics = regression_metrics(y, y_pred, n_features)
    metrics["Model"] = name

    return metrics


def plot_feature_importance(coefficients, pipeline):
    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

    coef_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients
    }).sort_values("Coefficient", key=abs, ascending=False)

    display(coef_df.head(10))
    fig = sns.barplot(
        data=coef_df,
        x='Coefficient',
        y='Feature'
    )
    # plt.title("Feature Importance", fontsize=14)
    # plt.xlabel("Coefficient Value", fontsize=12)
    plt.ylabel("Feature", fontsize=12)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.grid(True, alpha=0.3)
    return fig

In [ ]:
def plot_numerical_distribution(df, column):
    """
    Plots the histogram and a horizontal box plot for a given numerical column.
    """
    plt.figure(figsize=(12, 5))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(df[column].dropna(), kde=True)
    plt.title(f'Histogram of {column}')
    plt.xlabel(column)
    plt.ylabel('Frequency')

    # Horizontal Box Plot
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[column].dropna())
    plt.title(f'Box Plot of {column}')
    plt.xlabel(column)

    plt.tight_layout()
    plt.show()

def plot_categorical_distribution(df, column):
    """
    Plots the distribution of a given categorical column using a count plot.
    """
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x=column, palette='viridis')
    plt.title(f'Distribution of {column}')
    plt.xlabel(column)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 1. Data Loading and task definition

In [ ]:

df = pd.read_csv("AirQualityUCI.csv", sep=";", decimal=',')
df.head()


In [ ]:
TARGET_COL = ''

## 2. Exploratory Data Analysis

### Univariate analysis

Let's look on categorical features

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()

for col in categorical_cols:
    plot_categorical_distribution(df, col)


and let's look on numerical

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in numerical_cols:
    plot_numerical_distribution(df, col)

In [ ]:

sns.histplot(df[TARGET_COL], kde=True)
plt.title("Target Distribution")
plt.show()


## 3. Train / Test Split

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(f"Shape of data_X: {X.shape}")
print(f"Shape of data_y: {y.shape}")

# Split data into training and testing sets with stratification and shuffling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42, # for reproducibility
    shuffle=False,
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print('Train/Test split complete.')

## 4. Preprocessing

In [ ]:

# --------------------------------
# 1. Feature groups
# --------------------------------
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns
print("Numeric features: ", num_features)
print("Categorical features: ", cat_features)

# --------------------------------
# 2. Missing value handling
# --------------------------------
# Options for numeric features:
# - strategy="mean"
# - strategy="median"
# - strategy="most_frequent"
numeric_imputer = SimpleImputer(strategy="median")

# Options for categorical features:
# - strategy="most_frequent"
# - strategy="constant" (e.g., "missing")
categorical_imputer = SimpleImputer(strategy="most_frequent")

# --------------------------------
# 3. Numeric preprocessing
# --------------------------------
# Choose ONE scaler:
# - StandardScaler()  -> zero mean, unit variance (linear models)
# - MinMaxScaler()    -> bounded range [0, 1]
numeric_scaler = StandardScaler()
# numeric_scaler = MinMaxScaler()

numeric_pipeline = Pipeline(steps=[
    ("imputer", numeric_imputer),
    ("scaler", numeric_scaler)
])

# --------------------------------
# 4. Categorical preprocessing
# --------------------------------
# Encoding options:
# - OneHotEncoder -> linear models, no ordering assumption
# - OrdinalEncoder -> tree-based models or known ordering

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# categorical_encoder = OrdinalEncoder(
#     handle_unknown="use_encoded_value",
#     unknown_value=-1
# )

categorical_pipeline = Pipeline(steps=[
    ("imputer", categorical_imputer),
    ("encoder", categorical_encoder)
])

# --------------------------------
# 5. Column transformer
# --------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_features),
        ("cat", categorical_pipeline, cat_features)
    ],
    remainder="drop"  # or "passthrough"
)

# --------------------------------
# 6. (Optional) Interpolation for time series
# --------------------------------
# NOTE: interpolation is NOT part of sklearn pipelines.
# Apply it BEFORE train-test split to avoid leakage.

# Example:
# df[num_features] = df[num_features].interpolate(
#     method="linear",
#     limit_direction="both"
# )

## 5. Linear Regression

In [ ]:
pipe_linear = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
pipe_linear.fit(X_train, y_train)
y_lin_train = pipe_linear.predict(X_train)
y_lin_test = pipe_linear.predict(X_test)

In [ ]:
train_test_comparison(pipe_linear, y_train, y_test)

Let's check residuals

In [ ]:
plot_residuals(y_test, y_lin_test, "Linear Model Residuals")

Let's look on feature importance

In [ ]:
fig = plot_feature_importance(pipe_linear.named_steps["model"].coef_, pipe_linear)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Linear Regression", fontsize=14);

### with Regularization

In [ ]:
param_distributions = {
    "model": [Ridge(), Lasso(), ElasticNet()],
    "model__alpha": np.logspace(-4, 2, 50),
    # "model__l1_ratio": np.linspace(0, 1, 10)
}

search_linear = RandomizedSearchCV(
    pipe_linear,
    param_distributions,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

search_linear.fit(X_train, y_train)
best_linear = search_linear.best_estimator_
y_best_linear_train = best_linear.predict(X_train)
y_best_linear_pred = best_linear.predict(X_test)

In [ ]:
best_linear

In [ ]:
train_test_comparison(best_linear, y_train, y_test)

In [ ]:
comparison_df = pd.DataFrame([
    evaluate_model("best Linear Regression", best_linear, X_test, y_test),
    evaluate_model("Linear Regression", pipe_linear, X_test, y_test),
])

comparison_df.sort_values("RMSE", ascending=False)

In [ ]:
plot_residuals(y_test, y_best_linear_pred, "Best Linear Model Residuals")

In [ ]:
feature_names = best_linear.named_steps["preprocessor"].get_feature_names_out()
fig = plot_feature_importance(best_linear.named_steps["model"].coef_, pipe_linear)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Best Linear Regression", fontsize=14);

## 6. Gradient Boosting

In [ ]:
pipe_gb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))
])

gb_params = {
    "model__n_estimators": [100, 300, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__subsample": [0.6, 0.8, 1.0]
}

search_gb = RandomizedSearchCV(
    pipe_gb,
    gb_params,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

search_gb.fit(X_train, y_train)
best_gb = search_gb.best_estimator_
best_gb.predict(X_test)

In [ ]:
train_test_comparison(best_gb, y_train, y_test)

Let's check residuals

In [ ]:
plot_residuals(y_test, best_gb.predict(X_test), "Gradient Boosting Residuals")

Let's look on feature importance

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": best_gb.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(10)
plot_feature_importance(best_gb.named_steps["model"].feature_importances_, best_gb)
plt.xlabel("Feature Importance", fontsize=14);
plt.title("Feature Importance for Gradient Boosting Regression", fontsize=14);


# Comparison and Conclusion

In [ ]:
comparison_df = pd.DataFrame([
    evaluate_model("Linear Regression", best_linear, X_test, y_test),
    evaluate_model("Gradient Boosting", best_gb, X_test, y_test),
])

comparison_df.sort_values("RMSE")